In [2]:
#Лабораторная работа №7 Горьков Антон 5130901/30201
# Глава 7. Дискретное преобразование Фурье

In [3]:
import numpy as np
PI2 = 2 * np.pi

In [4]:
# Упражнение 7.2
#
# В этой главе показано, как выразить ДПФ и обратное ДПФ
# как произведения матриц. Время выполнения этих операций
# пропорционально N^2, где N — длина массива.
#
# Быстрое преобразование Фурье работает быстрее:
# его сложность порядка N log N.
#
# Ключевая идея — лемма Дэниелсона–Ланцоша:
#
# DFT(y)[n] = DFT(e)[n] + exp(-2 pi i n / N) DFT(o)[n]
#
# где e — четные элементы массива y,
# а o — нечетные элементы массива y.
#
# Это подсказывает рекурсивный алгоритм:
# 1. Разбить массив на четные и нечетные элементы.
# 2. Вычислить ДПФ обеих половин.
# 3. Объединить результаты по формуле выше.

In [5]:
# Для начала возьмем маленький тестовый сигнал.

ys = np.array([-0.5, 0.1, 0.7, -0.1])
hs = np.fft.fft(ys)
print(hs)

[ 0.2+0.j  -1.2-0.2j  0.2+0.j  -1.2+0.2j]


In [8]:
# Простая реализация ДПФ через матрицу.

def dft(ys):
    N = len(ys)
    ts = np.arange(N) / N
    freqs = np.arange(N)
    args = np.outer(ts, freqs)
    M = np.exp(1j * PI2 * args)
    amps = M.conj().transpose().dot(ys)
    return amps

In [9]:
# Проверим, что результат совпадает с np.fft.fft.

hs2 = dft(ys)
np.sum(np.abs(hs - hs2))

np.float64(5.290103952511234e-16)

In [11]:
# Первый шаг к БПФ:
# разбиваем массив на четные и нечетные элементы,
# но пока вычисляем их ДПФ обычной функцией dft.

def fft_norec(ys):
    N = len(ys)
    
    He = dft(ys[::2])
    Ho = dft(ys[1::2])
    
    ns = np.arange(N)
    W = np.exp(-1j * PI2 * ns / N)
    
    return np.tile(He, 2) + W * np.tile(Ho, 2)

In [12]:
# Проверим, что формула объединения работает правильно.

hs3 = fft_norec(ys)
np.sum(np.abs(hs - hs3))

np.float64(1.6653345369377348e-16)

In [13]:
# Та же промежуточная версия, но с np.fft.fft для четной и нечетной частей.
# Это удобно для отладки шага объединения.

def fft_norec2(ys):
    N = len(ys)
    
    He = np.fft.fft(ys[::2])
    Ho = np.fft.fft(ys[1::2])
    
    ns = np.arange(N)
    W = np.exp(-1j * PI2 * ns / N)
    
    return np.tile(He, 2) + W * np.tile(Ho, 2)

In [14]:
hs4 = fft_norec2(ys)
np.sum(np.abs(hs - hs4))

np.float64(0.0)

In [15]:
# Теперь заменяем вычисление половин на рекурсивные вызовы.
# Базовый случай: если длина массива 1, его ДПФ равен ему самому.

def fft(ys):
    N = len(ys)
    
    if N == 1:
        return ys
    
    He = fft(ys[::2])
    Ho = fft(ys[1::2])
    
    ns = np.arange(N)
    W = np.exp(-1j * PI2 * ns / N)
    
    return np.tile(He, 2) + W * np.tile(Ho, 2)

In [16]:
# Проверим на маленьком массиве.

hs5 = fft(ys)
np.sum(np.abs(hs - hs5))

np.float64(1.6653345369377348e-16)

In [17]:
# Проверим на случайном массиве длины 8.

ys = np.random.random(8)
hs = np.fft.fft(ys)
hs2 = fft(ys)

np.sum(np.abs(hs - hs2))

np.float64(2.034417050212713e-15)

In [18]:
# Проверим на нескольких размерах,
# которые являются степенями двойки.

for N in [2, 4, 8, 16, 32]:
    ys = np.random.random(N)
    hs = np.fft.fft(ys)
    hs2 = fft(ys)
    err = np.sum(np.abs(hs - hs2))
    print(N, err)

2 2.3189651794407217e-17
4 2.306439341124675e-16
8 2.4700641515208625e-15
16 6.248885254919425e-15
32 2.165711799743236e-14


In [21]:
#Вывод - функция также работает верно. fft и dft отличаются
#лишь тем, что fft работает быстрее, а именно за nlog(n), в то время как dft работает
#за n2